In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from config import SimConfig

In [2]:
# ==========================================
# 1. THE LOCKED PHYSICAL ANCHORS
# ==========================================
cfg = SimConfig()

P_MAX = cfg.p_max      # kW (Physical limit of module)
P_NOM = cfg.p_nom      # kW (Nominal wear baseline)
P_OPT = cfg.p_opt      # kW (Maximum efficiency point)
ETA_OPT = cfg.eta_opt    # 60% peak efficiency
K_FC = cfg.k_fc     # € (Module Capital Cost)
K_H2 = cfg.k_h2         # €/kg (Hydrogen Market Cost)
TAU_FC_S = cfg.tau_fc * 3600  # seconds (50,000 hours absolute lifespan)
LHV = cfg.LHV       # kWh/kg
K_E = 1000.0 / (LHV * 3600.0) # Conversion constant

P_D_MAX = 2400.0   # Max ship demand

# Helper function to prevent JSON errors in Plotly WebGL
def to_safe_list(arr):
    if not isinstance(arr, np.ndarray):
        arr = np.array(arr)
    arr_list = arr.tolist()
    if arr.ndim == 1:
        return [None if (isinstance(x, float) and np.isnan(x)) else x for x in arr_list]
    elif arr.ndim == 2:
        return [[None if (isinstance(x, float) and np.isnan(x)) else x for x in row] for row in arr_list]
    return arr_list

In [3]:
# ==========================================
# 2. GLOBAL UI (THE STEERING WHEEL)
# ==========================================
style = {'description_width': 'initial'}
center_layout = widgets.Layout(justify_content='center')

w_W_fuel = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.01, description='OPEX Blame Ratio (W_fuel):', style=style, layout=widgets.Layout(width='350px'))
w_tau_idle = widgets.FloatSlider(value=30, min=1, max=120, step=0.1, description='Idling Stubbornness τ_idle [min]:', style=style, layout=widgets.Layout(width='700px'))
w_kappa = widgets.FloatSlider(value=6.4, min=2.0, max=15.0, step=0.05, description='Stress Anchor κ_stress [Millions]:', style=style, layout=widgets.Layout(width='700px'))

global_controls = widgets.VBox([
    widgets.HTML("<h3 style='text-align:center;'>The Decoupled Control Space</h3><p style='text-align:center;'><i>Observe how adjusting W_fuel changes the bounds of the Stress Anchor to prevent physics violations.</i></p>"),
    widgets.HBox([w_W_fuel, w_tau_idle, w_kappa], layout=center_layout)
])

w_slice_pd = widgets.FloatSlider(value=1000.0, min=200.0, max=P_D_MAX, step=10.0, description='Slice Demand P_d [kW]:', style=style, layout=widgets.Layout(width='400px'))

In [4]:
# ==========================================
# 3. STATEFUL FIGURE INITIALIZATION
# ==========================================
# --- Tab 1: The Engine Room (Physical Curves & Gauges) ---
fig1 = go.FigureWidget(make_subplots(
    rows=1, cols=3, column_widths=[0.6, 0.2, 0.2],
    specs=[[{"type": "xy"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=("Derived Thermodynamic Efficiency Curve", "Mechanical Limit", "Degradation Rate")
))
fig1.add_trace(go.Scatter(name="η(p)", line=dict(color='#2ca02c', width=4)), row=1, col=1)
fig1.add_trace(go.Scatter(
    mode='markers+text', name="Anchors", marker=dict(size=12, color='black'), 
    textposition="top center",
    hovertemplate="Power: %{x:.1f} kW<br>Efficiency: %{y:.3f}<extra></extra>"
), row=1, col=1)
fig1.add_trace(go.Indicator(mode="number+gauge", title={'text': "S_max"}, gauge={'axis': {'range': [0, 15000]}, 'bar': {'color': "#1f77b4"}}), row=1, col=2)
fig1.add_trace(go.Indicator(mode="number+gauge", title={'text': "Alpha (α)"}, number={'valueformat': ".2f"}, gauge={'axis': {'range': [0, 4.0]}, 'bar': {'color': "#ff7f0e"}}), row=1, col=3)
fig1.update_layout(height=500, width=1350, plot_bgcolor='rgba(240, 240, 240, 1)')
fig1.update_xaxes(title_text="Module Power (kW)", row=1, col=1)
fig1.update_yaxes(title_text="Efficiency (η)", row=1, col=1, range=[0.3, 0.65])

# --- Tab 2: The Brain (Break-Even Map) ---
fig2 = go.FigureWidget()
fig2.add_trace(go.Heatmap(colorscale='Plasma', zmin=0, zmax=8.0, colorbar=dict(title="Hours")))
fig2.add_trace(go.Scatter(name="Ideal Continuous (n*)", line=dict(color='white', dash='dash', width=2)))
fig2.add_trace(go.Scatter(name="Ideal Discrete", line=dict(color='cyan', width=2, shape='hv')))
fig2.update_layout(height=650, width=1350, plot_bgcolor='rgba(240, 240, 240, 1)', xaxis_title="Power Demand P_d (kW)", yaxis_title="Active Modules (n)", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig2.update_yaxes(range=[0.5, 16.5], tickmode='linear', tick0=1, dtick=1)

# --- Tab 3: The Penalty (Cost Landscapes) ---
fig3 = go.FigureWidget(make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35], subplot_titles=("Operational Penalty Heatmap", "1D Penalty Slice")))
fig3.add_trace(go.Heatmap(colorscale='Viridis', zmin=0, zmax=30.0, colorbar=dict(title="% Penalty", x=0.6)), row=1, col=1)
fig3.add_trace(go.Scatter(name="Ideal Continuous (n*)", line=dict(color='white', dash='dash', width=2)), row=1, col=1)
fig3.add_trace(go.Scatter(name="Ideal Discrete", line=dict(color='red', width=2, shape='hv')), row=1, col=1)
fig3.add_trace(go.Scatter(name="Penalty %", line=dict(color='purple', width=3)), row=1, col=2)
fig3.add_trace(go.Scatter(mode='markers', name="Optimal", marker=dict(color='red', size=12, symbol='star')), row=1, col=2)
fig3.update_layout(height=600, width=1350, plot_bgcolor='rgba(240, 240, 240, 1)', legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.3))
fig3.update_xaxes(title_text="Power Demand P_d (kW)", row=1, col=1)
fig3.update_yaxes(title_text="Active Modules (n)", row=1, col=1, range=[0.5, 16.5], tickmode='linear', dtick=1)
fig3.update_xaxes(title_text="Active Modules (n)", row=1, col=2, range=[0.5, 16.5], tickmode='linear', dtick=2)
fig3.update_yaxes(title_text="Cost Penalty (%)", row=1, col=2, range=[0, 30])

FigureWidget({
    'data': [{'colorbar': {'title': {'text': '% Penalty'}, 'x': 0.6},
              'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'type': 'heatmap',
              'uid': '9ca1bfc7-0d7e-46c4-b5aa-1653fc8910eb',
              'xaxis': 'x',
              'yaxis': 'y',
              'zmax': 30.0,
              'zmin': 0},
             {'line': {'color': 'white', 'dash': 'dash', 'width': 2},
              'name': 'Ideal Continuous (n*)',
              'type': 'scatter',
              'uid': '752cac74-8755-4b17-a8b4-42955be5a2b8',
              'xaxis': 'x

In [5]:
# ==========================================
# 4. OBSERVER LOGIC & BATCH UPDATING
# ==========================================
def master_observer(*args):
    W = w_W_fuel.value
    slice_pd = w_slice_pd.value
    tau_idle = w_tau_idle.value * 60
    
# 1. ANALYTICAL PHYSICAL LIMITS
    p_star_min = (W / (P_OPT**2) + (1 - W) / (P_NOM**2))**-0.5
    p_star_max = P_OPT / np.sqrt(W)
    
    # Ceil the min and floor the max to 1 decimal place to strictly avoid singularities
    raw_kappa_min = tau_idle * (p_star_min**2) / 1e6
    raw_kappa_max = tau_idle * (p_star_max**2) / 1e6
    
    kappa_min = np.ceil(raw_kappa_min * 10) / 10.0
    kappa_max = np.floor(raw_kappa_max * 10) / 10.0
    
    if kappa_max < w_kappa.min:
        w_kappa.min, w_kappa.max = kappa_min, kappa_max
    else:
        w_kappa.max, w_kappa.min = kappa_max, kappa_min
        
    w_kappa.value = max(kappa_min, min(kappa_max, w_kappa.value))
    
    # 2. REVERSE-ENGINEER THE PHYSICS ENGINE
    kappa_stress = w_kappa.value * 1e6
    p_star = np.sqrt(kappa_stress / tau_idle)
    
    bracket = max(1e-9, W / (P_OPT**2) + (1 - W) / (P_NOM**2) - 1 / (p_star**2))
    S_max_raw = (TAU_FC_S * P_NOM**2 / tau_idle) * bracket
    S_max = max(1, round(S_max_raw))
    
    alpha = max(0.0, (1 - W) * (TAU_FC_S / (S_max * tau_idle)) - 1)
    
    a2 = (W * K_FC) / (S_max * tau_idle * P_OPT**2 * (K_H2 / 1000))
    a0 = a2 * P_OPT**2
    a1 = (K_E / ETA_OPT) - 2 * a2 * P_OPT
    
    A = a2 * K_H2 / 1000 + (alpha * K_FC) / (TAU_FC_S * P_NOM**2)
    B = a1 * K_H2 / 1000 - (2 * K_FC * alpha) / (TAU_FC_S * P_NOM)
    C = a0 * K_H2 / 1000 + (K_FC * (1 + alpha)) / TAU_FC_S
    k_s = K_FC / S_max

    # 3. UPDATE TAB 1 (THE ENGINE ROOM)
    p_arr = np.linspace(15, P_MAX, 150)
    eta_arr = (K_E * p_arr) / (a2 * p_arr**2 + a1 * p_arr + a0)
    
    with fig1.batch_update():
        fig1.data[0].x, fig1.data[0].y = to_safe_list(p_arr), to_safe_list(eta_arr)
        fig1.data[1].x = [P_OPT, p_star, P_NOM]
        fig1.data[1].y = [ETA_OPT, (K_E * p_star) / (a2*p_star**2 + a1*p_star + a0), (K_E * P_NOM) / (a2*P_NOM**2 + a1*P_NOM + a0)]
        fig1.data[1].text = ["p_opt", "<b>p*</b>", "p_nom"] 
        fig1.data[2].value, fig1.data[3].value = S_max, alpha
        fig1.layout.title.text = f"<span style='font-size:18px'><b>Target Sweet Spot: p* = {p_star:.1f} kW</b></span>"

    # 4. UPDATE TAB 2 & 3 (THE BRAIN & PENALTY)
    P_d_grid = np.linspace(200, P_D_MAX, 150)
    N_grid = np.arange(1, 17)
    PP, NN = np.meshgrid(P_d_grid, N_grid)
    
    Co = A * (PP**2 / NN) + B * PP + NN * C
    n_opt_disc = N_grid[np.argmin(Co, axis=0)]
    Co_opt_disc = np.min(Co, axis=0)
    
    n_star_cont = P_d_grid / p_star
    savings_per_sec = Co - np.tile(Co_opt_disc, (len(N_grid), 1))
    switch_cost_total = k_s * np.abs(NN - np.tile(n_opt_disc, (len(N_grid), 1)))
    
    with np.errstate(divide='ignore', invalid='ignore'):
        T_be_hours = np.where(savings_per_sec > 1e-6, (switch_cost_total / savings_per_sec) / 3600.0, np.nan)
        delta_C_percent = 100 * savings_per_sec / np.tile(Co_opt_disc, (len(N_grid), 1))
    
    idx_slice = np.abs(P_d_grid - slice_pd).argmin()
    actual_slice_pd = P_d_grid[idx_slice]
    slice_percent = delta_C_percent[:, idx_slice]
    best_n_slice = n_opt_disc[idx_slice]

    with fig2.batch_update():
        fig2.data[0].x, fig2.data[0].y = to_safe_list(P_d_grid), to_safe_list(N_grid)
        fig2.data[0].z = to_safe_list(T_be_hours)
        fig2.data[1].x, fig2.data[1].y = to_safe_list(P_d_grid), to_safe_list(n_star_cont)
        fig2.data[2].x, fig2.data[2].y = to_safe_list(P_d_grid), to_safe_list(n_opt_disc)
        fig2.layout.title.text = f"<span style='font-size:18px'><b>Absolute Controller Deadband</b> | Base Switch Cost: {k_s:.2f} €</span>"

    with fig3.batch_update():
        fig3.data[0].x, fig3.data[0].y = to_safe_list(P_d_grid), to_safe_list(N_grid)
        fig3.data[0].z = to_safe_list(delta_C_percent)
        fig3.data[1].x, fig3.data[1].y = to_safe_list(P_d_grid), to_safe_list(n_star_cont)
        fig3.data[2].x, fig3.data[2].y = to_safe_list(P_d_grid), to_safe_list(n_opt_disc)
        fig3.data[3].x, fig3.data[3].y = to_safe_list(N_grid), to_safe_list(slice_percent)
        fig3.data[4].x, fig3.data[4].y = [best_n_slice], [0]
        fig3.layout.annotations[1].text = f"1D Penalty Slice at P_d = {actual_slice_pd:.0f} kW"
        
# Bind observers
for w in [w_W_fuel, w_tau_idle, w_kappa, w_slice_pd]:
    w.observe(master_observer, 'value')

In [6]:
# ==========================================
# 5. CONSTRUCT LAYOUT
# ==========================================
tab1 = widgets.VBox([fig1])
tab2 = widgets.VBox([fig2])
tab3 = widgets.VBox([widgets.HBox([w_slice_pd], layout=center_layout), fig3])

dashboard = widgets.Tab(children=[tab1, tab2, tab3])
dashboard.set_title(0, '1. The Engine Room (Physics)')
dashboard.set_title(1, '2. The Controller Brain (Break-Even)')
dashboard.set_title(2, '3. The Penalty Landscape')

master_observer()
display(widgets.VBox([global_controls, dashboard]))